# YoungXCode — Team Productivity & Workload Analysis Report
### Real Data from YoungXCode Business Dataset
**Team Members:** Sneha Kapoor, Vikram Singh, Ananya Joshi, Karan Malhotra, Rohan Gupta, Priya Mehta, Isha Patel, Siddharth Das, Arjun Sharma, Divya Nair, Pooja Reddy, Rahul Verma  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Formula:** Team Utilization = (Total Actual Hours ÷ Total Available Hours) × 100


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#f9f9f9',
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
})
print('Libraries loaded.')


## 1. Load Real YoungXCode Dataset

In [ ]:
import openpyxl

FILE = '/mnt/user-data/uploads/1781005645430_YoungXCode_BusinessDataset__4_.xlsx'

tasks_df    = pd.read_excel(FILE, sheet_name='Tasks')
projects_df = pd.read_excel(FILE, sheet_name='Projects')

# Parse dates
for col in ['Created Date','Completed Date']:
    tasks_df[col] = pd.to_datetime(tasks_df[col], errors='coerce')
for col in ['Start Date','Deadline','Actual End Date']:
    projects_df[col] = pd.to_datetime(projects_df[col], errors='coerce')

# Clean column names
tasks_df.columns    = tasks_df.columns.str.strip()
projects_df.columns = projects_df.columns.str.strip()

# Flags
tasks_df['is_completed'] = tasks_df['Completed Date'].notna()
tasks_df['task_days']    = (tasks_df['Completed Date'] - tasks_df['Created Date']).dt.days
tasks_df['hours_ratio']  = tasks_df['Actual Hours'] / tasks_df['Estimated Hours']

# Overdue = not completed (no Completed Date) and task is old (created > 30 days ago)
today = pd.Timestamp('2025-01-15')
tasks_df['is_overdue'] = (
    ~tasks_df['is_completed'] &
    ((today - tasks_df['Created Date']).dt.days > 30)
)

print(f'Tasks loaded      : {len(tasks_df)}')
print(f'Team members      : {tasks_df["Assignee"].nunique()}')
print(f'Projects in tasks : {tasks_df["Project Name"].nunique()}')
print(f'Completed tasks   : {tasks_df["is_completed"].sum()}')
print(f'Open/overdue tasks: {tasks_df["is_overdue"].sum()}')
tasks_df.head()


## 2. Per-Member Productivity Metrics

In [ ]:
AVAILABLE_HOURS_PER_PERSON = 160  # 8 hrs x 20 working days per month

member_stats = tasks_df.groupby('Assignee').agg(
    tasks_assigned      = ('Project Name', 'count'),
    tasks_completed     = ('is_completed', 'sum'),
    overdue_tasks       = ('is_overdue', 'sum'),
    total_est_hours     = ('Estimated Hours', 'sum'),
    total_actual_hours  = ('Actual Hours', lambda x: x.dropna().sum()),
    avg_task_days       = ('task_days', 'mean'),
).reset_index()

member_stats['completion_rate']  = (member_stats['tasks_completed'] / member_stats['tasks_assigned'] * 100).round(1)
member_stats['hours_ratio']      = (member_stats['total_actual_hours'] / member_stats['total_est_hours']).round(2)
member_stats['utilization_pct']  = (member_stats['total_actual_hours'] / AVAILABLE_HOURS_PER_PERSON * 100).round(1)
member_stats['avg_task_days']    = member_stats['avg_task_days'].round(1)

# Flags
member_stats['is_overallocated'] = member_stats['utilization_pct'] > 100
member_stats['is_underutilized'] = member_stats['utilization_pct'] < 50

pd.set_option('display.float_format', '{:.1f}'.format)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 180)

print('TEAM PRODUCTIVITY TABLE (sorted by completion rate):')
print('=' * 130)
print(f'  {"Assignee":<20} {"Assigned":>9} {"Completed":>10} {"Rate%":>7} {"Overdue":>8} '
      f'{"Est Hrs":>8} {"Act Hrs":>8} {"Ratio":>7} {"Avg Days":>9} {"Utiliz%":>9}')
print('-' * 130)
for _, r in member_stats.sort_values('completion_rate', ascending=False).iterrows():
    flag = ' [OVER]' if r['is_overallocated'] else ' [UNDER]' if r['is_underutilized'] else ''
    print(f'  {r["Assignee"]:<20} {r["tasks_assigned"]:>9} {r["tasks_completed"]:>10.0f} '
          f'{r["completion_rate"]:>7.1f} {r["overdue_tasks"]:>8.0f} '
          f'{r["total_est_hours"]:>8.1f} {r["total_actual_hours"]:>8.1f} '
          f'{r["hours_ratio"]:>7.2f} {r["avg_task_days"]:>9.1f} {r["utilization_pct"]:>9.1f}{flag}')
print('=' * 130)


## 3. Overall Team Utilization Rate

In [ ]:
n_members          = member_stats['Assignee'].nunique()
total_actual_hrs   = member_stats['total_actual_hours'].sum()
total_avail_hrs    = n_members * AVAILABLE_HOURS_PER_PERSON
team_utilization   = (total_actual_hrs / total_avail_hrs) * 100

total_tasks        = member_stats['tasks_assigned'].sum()
total_completed    = member_stats['tasks_completed'].sum()
overall_completion = total_completed / total_tasks * 100
overallocated_cnt  = member_stats['is_overallocated'].sum()
underutilized_cnt  = member_stats['is_underutilized'].sum()

print('=' * 55)
print('   TEAM UTILIZATION & OVERALL KPIs')
print('=' * 55)
print(f'  Team Members           : {n_members}')
print(f'  Total Available Hours  : {total_avail_hrs:.0f} hrs')
print(f'  Total Actual Hours     : {total_actual_hrs:.1f} hrs')
print(f'  Team Utilization Rate  : {team_utilization:.1f}%')
print(f'  Formula                : ({total_actual_hrs:.0f} / {total_avail_hrs}) x 100')
print('-' * 55)
print(f'  Total Tasks Assigned   : {total_tasks}')
print(f'  Total Tasks Completed  : {total_completed:.0f}')
print(f'  Overall Completion Rate: {overall_completion:.1f}%')
print(f'  Over-Allocated Members : {overallocated_cnt}')
print(f'  Under-Utilized Members : {underutilized_cnt}')
print('=' * 55)


## 4. Busiest vs Least Busy Team Members

In [ ]:
busiest = member_stats.sort_values('total_actual_hours', ascending=False)
top5    = busiest.head(5)
bottom5 = busiest.tail(5)

print('TOP 5 BUSIEST (by actual hours logged):')
print(top5[['Assignee','tasks_assigned','tasks_completed','total_actual_hours','utilization_pct']].to_string(index=False))
print()
print('TOP 5 LEAST BUSY (available capacity):')
print(bottom5[['Assignee','tasks_assigned','tasks_completed','total_actual_hours','utilization_pct']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Busiest vs Least Busy Team Members', fontsize=15, fontweight='bold')

sorted_util = member_stats.sort_values('utilization_pct', ascending=True)
util_colors = ['#B71C1C' if v > 100 else '#1B5E20' if 75 <= v <= 100 else '#F9A825' if 50 <= v < 75 else '#90A4AE'
               for v in sorted_util['utilization_pct']]
bars = axes[0].barh(sorted_util['Assignee'], sorted_util['utilization_pct'], color=util_colors, edgecolor='white')
for bar, val in zip(bars, sorted_util['utilization_pct']):
    axes[0].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'{val:.0f}%', va='center', fontsize=9, fontweight='bold')
axes[0].axvline(x=75, color='green', linestyle='--', linewidth=1.2, label='Min target 75%')
axes[0].axvline(x=100, color='red',   linestyle='--', linewidth=1.2, label='Burnout risk 100%')
axes[0].set_xlabel('Utilization (%)')
axes[0].set_title('Team Utilization Rate per Member', fontsize=12)
axes[0].set_xlim(0, 130)
axes[0].legend(fontsize=9)

sorted_tasks = member_stats.sort_values('tasks_assigned', ascending=True)
t_colors = ['#1565C0' if v >= 20 else '#42A5F5' if v >= 12 else '#90CAF9' for v in sorted_tasks['tasks_assigned']]
bars2 = axes[1].barh(sorted_tasks['Assignee'], sorted_tasks['tasks_assigned'], color=t_colors, edgecolor='white')
for bar, comp, tot in zip(bars2, sorted_tasks['tasks_completed'], sorted_tasks['tasks_assigned']):
    axes[1].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                 f'{comp:.0f}/{tot} done', va='center', fontsize=9)
axes[1].set_xlabel('Tasks Assigned')
axes[1].set_title('Total Tasks Assigned per Member', fontsize=12)

plt.tight_layout()
plt.savefig('utilization_tasks.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Over-Allocated & Burnout Risk Detection

In [ ]:
# Risk scoring: utilization > 100 OR hours_ratio > 1.2 OR overdue > 3
member_stats['risk_score'] = (
    (member_stats['utilization_pct'] > 100).astype(int) * 3 +
    (member_stats['hours_ratio'] > 1.2).astype(int) * 2 +
    (member_stats['overdue_tasks'] > 3).astype(int) * 2 +
    (member_stats['completion_rate'] < 60).astype(int) * 1
)

def risk_level(score):
    if score >= 5: return 'HIGH RISK'
    if score >= 3: return 'MODERATE'
    return 'HEALTHY'

member_stats['risk_level'] = member_stats['risk_score'].apply(risk_level)

risk_sorted = member_stats.sort_values('risk_score', ascending=False)
print('OVER-ALLOCATION & BURNOUT RISK REPORT:')
print(risk_sorted[['Assignee','utilization_pct','hours_ratio','overdue_tasks','completion_rate','risk_score','risk_level']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
risk_colors_map = {'HIGH RISK':'#B71C1C','MODERATE':'#F57F17','HEALTHY':'#2E7D32'}
bar_colors3 = [risk_colors_map[r] for r in risk_sorted['risk_level']]
bars3 = ax.bar(risk_sorted['Assignee'], risk_sorted['risk_score'], color=bar_colors3, edgecolor='white')
for bar, val, rl in zip(bars3, risk_sorted['risk_score'], risk_sorted['risk_level']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{val}\n{rl}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Risk Score')
ax.set_title('Burnout & Over-Allocation Risk per Team Member', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=30)

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in risk_colors_map.items()]
ax.legend(handles=legend_patches)
plt.tight_layout()
plt.savefig('burnout_risk.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Weekly Productivity Trend

In [ ]:
weekly = tasks_df.dropna(subset=['Created Date']).copy()
weekly['week'] = weekly['Created Date'].dt.to_period('W')

weekly_agg = weekly.groupby('week').agg(
    tasks_created    = ('Project Name','count'),
    tasks_completed  = ('is_completed','sum'),
    actual_hours     = ('Actual Hours', lambda x: x.dropna().sum()),
    est_hours        = ('Estimated Hours', lambda x: x.dropna().sum()),
).reset_index()

weekly_agg['week_str'] = weekly_agg['week'].astype(str)
weekly_agg['completion_rate'] = (weekly_agg['tasks_completed'] / weekly_agg['tasks_created'] * 100).round(1)

# Plot last 20 weeks for clarity
plot_w = weekly_agg.tail(20).reset_index(drop=True)
x = np.arange(len(plot_w))

fig, ax1 = plt.subplots(figsize=(15, 6))
ax1.bar(x-0.2, plot_w['tasks_created'],   0.35, label='Tasks Created',   color='#1565C0', alpha=0.85)
ax1.bar(x+0.2, plot_w['tasks_completed'], 0.35, label='Tasks Completed', color='#2E7D32', alpha=0.85)

ax2 = ax1.twinx()
ax2.plot(x, plot_w['actual_hours'], color='#E65100', marker='o', linewidth=2, markersize=5,
         label='Actual Hours Logged')
ax2.set_ylabel('Hours Logged', color='#E65100', fontsize=10)
ax2.tick_params(axis='y', labelcolor='#E65100')

ax1.set_xticks(x)
ax1.set_xticklabels([w[-5:] for w in plot_w['week_str']], rotation=45, ha='right', fontsize=8)
ax1.set_xlabel('Week')
ax1.set_ylabel('Task Count')
ax1.set_title('Weekly Productivity Trend (Last 20 Weeks)', fontsize=13, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('weekly_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Task Distribution by Role / Priority

In [ ]:
# Infer roles based on known context (YoungXCode team)
role_map = {
    'Arjun Sharma':    'Senior Developer',
    'Priya Mehta':     'Developer',
    'Vikram Singh':    'Developer',
    'Rohan Gupta':     'Developer',
    'Sneha Kapoor':    'Designer',
    'Ananya Joshi':    'Designer',
    'Karan Malhotra':  'DevOps Engineer',
    'Siddharth Das':   'QA Engineer',
    'Divya Nair':      'Project Manager',
    'Isha Patel':      'Frontend Developer',
    'Pooja Reddy':     'Data Analyst',
    'Rahul Verma':     'Backend Developer',
}
tasks_df['Role'] = tasks_df['Assignee'].map(role_map)

role_stats = tasks_df.groupby('Role').agg(
    tasks_assigned   = ('Project Name','count'),
    tasks_completed  = ('is_completed','sum'),
    actual_hours     = ('Actual Hours', lambda x: x.dropna().sum()),
    overdue          = ('is_overdue','sum'),
).reset_index()
role_stats['completion_rate'] = (role_stats['tasks_completed'] / role_stats['tasks_assigned'] * 100).round(1)
role_stats = role_stats.sort_values('tasks_assigned', ascending=False)

print('Task Distribution by Role:')
print(role_stats.to_string(index=False))

priority_counts = tasks_df['Priority'].value_counts()
priority_comp   = tasks_df.groupby('Priority')['is_completed'].mean() * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Task Distribution by Role & Priority', fontsize=15, fontweight='bold')

# Role tasks bar
role_colors = sns.color_palette('Blues_d', len(role_stats))
bars = axes[0].bar(role_stats['Role'], role_stats['tasks_assigned'], color=role_colors, edgecolor='white')
for bar, val in zip(bars, role_stats['tasks_assigned']):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_ylabel('Tasks Assigned')
axes[0].set_title('Tasks by Role', fontsize=12)
axes[0].tick_params(axis='x', rotation=40)

# Role completion rate
rate_colors = ['#1B5E20' if v >= 70 else '#F9A825' if v >= 50 else '#B71C1C' for v in role_stats['completion_rate']]
bars2 = axes[1].bar(role_stats['Role'], role_stats['completion_rate'], color=rate_colors, edgecolor='white')
for bar, val in zip(bars2, role_stats['completion_rate']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_ylabel('Completion Rate (%)')
axes[1].set_title('Completion Rate by Role', fontsize=12)
axes[1].tick_params(axis='x', rotation=40)

# Priority pie
pri_colors = {'Critical':'#B71C1C','High':'#E65100','Medium':'#1565C0','Low':'#2E7D32'}
clrs = [pri_colors.get(p,'gray') for p in priority_counts.index]
wedges, texts, autotexts = axes[2].pie(
    priority_counts.values, labels=priority_counts.index,
    autopct='%1.1f%%', colors=clrs,
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
axes[2].set_title('Task Priority Distribution', fontsize=12)

plt.tight_layout()
plt.savefig('role_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Individual Completion Rate & Hours Ratio Chart

In [ ]:
sorted_comp = member_stats.sort_values('completion_rate', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Individual Completion Rate & Hours Ratio', fontsize=15, fontweight='bold')

comp_colors = ['#1B5E20' if v >= 70 else '#F9A825' if v >= 50 else '#B71C1C'
               for v in sorted_comp['completion_rate']]
bars = axes[0].barh(sorted_comp['Assignee'], sorted_comp['completion_rate'],
                    color=comp_colors, edgecolor='white')
for bar, val in zip(bars, sorted_comp['completion_rate']):
    axes[0].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
axes[0].axvline(x=70, color='navy', linestyle='--', linewidth=1.2, label='Target 70%')
axes[0].set_xlabel('Task Completion Rate (%)')
axes[0].set_title('Task Completion Rate per Member', fontsize=12)
axes[0].set_xlim(0, 110)
axes[0].legend(fontsize=9)

sorted_ratio = member_stats.sort_values('hours_ratio', ascending=True)
ratio_colors = ['#B71C1C' if v > 1.2 else '#1B5E20' if v <= 1.0 else '#F9A825'
                for v in sorted_ratio['hours_ratio']]
bars2 = axes[1].barh(sorted_ratio['Assignee'], sorted_ratio['hours_ratio'],
                     color=ratio_colors, edgecolor='white')
for bar, val in zip(bars2, sorted_ratio['hours_ratio']):
    axes[1].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                 f'{val:.2f}x', va='center', fontsize=9, fontweight='bold')
axes[1].axvline(x=1.0, color='green', linestyle='--', linewidth=1.2, label='Ideal ratio 1.0')
axes[1].axvline(x=1.2, color='red',   linestyle='--', linewidth=1.2, label='Overrun > 1.2x')
axes[1].set_xlabel('Actual / Estimated Hours Ratio')
axes[1].set_title('Hours Ratio (Actual / Estimated)', fontsize=12)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('completion_hours_ratio.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Key Findings

In [ ]:
top_busy     = member_stats.sort_values('total_actual_hours', ascending=False).iloc[0]
top_idle     = member_stats.sort_values('total_actual_hours').iloc[0]
best_compl   = member_stats.sort_values('completion_rate', ascending=False).iloc[0]
worst_compl  = member_stats.sort_values('completion_rate').iloc[0]
most_overdue = member_stats.sort_values('overdue_tasks', ascending=False).iloc[0]

print('=' * 60)
print('   KEY FINDINGS — YoungXCode Team Productivity')
print('=' * 60)
print(f'  1. Team Utilization Rate   : {team_utilization:.1f}%  (Target: 75-85%)')
print(f'  2. Overall Completion Rate : {overall_completion:.1f}%  ({total_completed:.0f}/{total_tasks} tasks)')
print(f'  3. Busiest Member          : {top_busy["Assignee"]} ({top_busy["total_actual_hours"]:.0f} hrs logged)')
print(f'  4. Least Busy Member       : {top_idle["Assignee"]} ({top_idle["total_actual_hours"]:.0f} hrs logged)')
print(f'  5. Best Completion Rate    : {best_compl["Assignee"]} ({best_compl["completion_rate"]:.1f}%)')
print(f'  6. Lowest Completion Rate  : {worst_compl["Assignee"]} ({worst_compl["completion_rate"]:.1f}%)')
print(f'  7. Most Overdue Tasks      : {most_overdue["Assignee"]} ({most_overdue["overdue_tasks"]:.0f} overdue tasks)')
print(f'  8. Over-Allocated Members  : {overallocated_cnt} members exceed 100% utilization')
print(f'  9. Under-Utilized Members  : {underutilized_cnt} members below 50% utilization')
print(f' 10. High-Risk Burnout Count : {(member_stats["risk_level"]=="HIGH RISK").sum()} members flagged HIGH RISK')
print('=' * 60)


## 10. Actionable Recommendations

In [ ]:
print('=' * 60)
print('   ACTIONABLE RECOMMENDATIONS')
print('=' * 60)
print()
print('REC 1 — Rebalance Workload from Overloaded to Idle Members')
print('-' * 60)
print(f'  {top_busy["Assignee"]} is carrying the heaviest load.')
print(f'  Redistribute tasks to {top_idle["Assignee"]} and other under-utilized')
print('  members. Target: keep every member between 75-95% utilization.')
print()
print('REC 2 — Improve Completion Rate for Low Performers')
print('-' * 60)
print(f'  {worst_compl["Assignee"]} has the lowest completion rate ({worst_compl["completion_rate"]:.1f}%).')
print('  Schedule a 1:1 review to identify blockers. Assign a buddy')
print('  from high-performing members for task guidance.')
print()
print('REC 3 — Reduce Overdue Tasks Immediately')
print('-' * 60)
print(f'  {most_overdue["Assignee"]} has {most_overdue["overdue_tasks"]:.0f} overdue tasks.')
print('  Run a weekly overdue task review every Monday. Any task')
print('  open > 14 days must be escalated or re-assigned.')
print()
print('REC 4 — Fix Hours Estimation Accuracy')
print('-' * 60)
print('  Several members show hours ratio > 1.2x (actual > estimated).')
print('  Introduce a retrospective hours log after each task.')
print('  Use historical actuals to improve future estimates.')
print()
print('REC 5 — Burnout Prevention for High-Risk Members')
print('-' * 60)
high_risk_names = member_stats[member_stats["risk_level"]=="HIGH RISK"]["Assignee"].tolist()
print(f'  High-risk members: {", ".join(high_risk_names) if high_risk_names else "None currently"}')
print('  Cap new task assignment for HIGH RISK members until their')
print('  current queue drops below 5 active tasks.')
print()
print('REC 6 — Weekly Capacity Planning Reviews')
print('-' * 60)
print(f'  Current team utilization: {team_utilization:.1f}%.')
print('  Hold a 30-min weekly capacity meeting every Monday.')
print('  Share a live utilization dashboard with all team leads to')
print('  proactively manage assignments and avoid bottlenecks.')
print('=' * 60)


## 11. Export Results to CSV

In [ ]:
member_stats.to_csv('team_productivity_report.csv', index=False)
role_stats.to_csv('role_distribution.csv', index=False)
tasks_df[tasks_df['is_overdue']].to_csv('overdue_tasks.csv', index=False)
weekly_agg.to_csv('weekly_trend.csv', index=False)

print('Exported:')
print('  -> team_productivity_report.csv')
print('  -> role_distribution.csv')
print('  -> overdue_tasks.csv')
print('  -> weekly_trend.csv')
